In [37]:
# Shared setup — run this cell first.
# Flip USE_LOCAL_LLM to choose the backend for both LangChain (`llm`) and DeepEval (`judge`).
import os
from pathlib import Path
from dotenv import load_dotenv

USE_LOCAL_LLM = True  # True = Ollama (local), False = OpenAI

OPENAI_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = "qwen3:8b"
OLLAMA_BASE_URL = "http://localhost:11434"

cwd = Path.cwd()
env_candidates = [
    cwd / ".env",
    cwd / "notebooks" / ".env",
    cwd.parent / "notebooks" / ".env",
    cwd.parent / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)
if env_file is None:
    raise FileNotFoundError("No .env found. Expected notebooks/.env with API keys.")
load_dotenv(env_file, override=True)

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")
os.environ.setdefault("DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE", "600")
os.environ["DEEPEVAL_DISABLE_DOTENV"] = "1"
os.environ.setdefault("DEEPEVAL_RESULTS_FOLDER", "./deepeval-results")


def _require_env(name: str) -> str:
    value = (os.getenv(name) or "").strip()
    if not value or "paste_your_key" in value:
        raise ValueError(f"Set {name} in {env_file}.")
    return value


if USE_LOCAL_LLM:
    from langchain_ollama import ChatOllama
    from deepeval.models import OllamaModel

    llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.5)
    judge = OllamaModel(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.0)
    backend = f"local Ollama ({OLLAMA_MODEL} @ {OLLAMA_BASE_URL})"
else:
    _require_env("OPENAI_API_KEY")
    from langchain_openai import ChatOpenAI
    from deepeval.models import OpenAIModel

    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.5)
    judge = OpenAIModel(model=OPENAI_MODEL, temperature=0.0)
    backend = f"OpenAI ({OPENAI_MODEL})"

print(f"Loaded env from: {env_file}")
print(f"Evaluation backend: {backend}")
print(f"OPENAI_API_KEY configured: {bool(os.getenv('OPENAI_API_KEY'))}")
if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")


Loaded env from: c:\Users\Girish Kulkarni\OneDrive\Documents\LLM_Testing\notebooks\.env
Evaluation backend: local Ollama (qwen3:8b @ http://localhost:11434)
OPENAI_API_KEY configured: True
LangSmith tracing enabled for project: LangChainTrainings-Agents


In [38]:
# Smoke-test the LLM created in the setup cell
llm.invoke("Hello, how are you?")

AIMessage(content="Hello! I'm just a chatbot, so I don't have feelings, but I'm here and ready to help! 😊 How are you doing today? Let me know if you need anything!", additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-09-20T04:49:40.3510733Z', 'done': True, 'done_reason': 'stop', 'total_duration': 34378104000, 'load_duration': 17053027300, 'prompt_eval_count': 16, 'prompt_eval_duration': 528197000, 'eval_count': 149, 'eval_duration': 16785930000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a0bd25-75c2-7f72-90eb-da116c5f143e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 149, 'total_tokens': 165})

In [39]:
# DeepEval status (uses `judge` from the setup cell)
import deepeval
from deepeval.confident.api import get_confident_api_key

print(f"DeepEval version: {__import__('importlib.metadata').metadata.version('deepeval')}")
print(f"Confident AI key configured: {get_confident_api_key() is not None}")
print(f"Judge model: {judge}")


DeepEval version: 4.2.3
Confident AI key configured: True
Judge model: <deepeval.models.llms.ollama_model.OllamaModel object at 0x000001A28CF1A2D0>


In [40]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

answer_relevancy_metric = AnswerRelevancyMetric(model=judge)

test_case = LLMTestCase(
    input="Capital of India",
    actual_output="Delhi",
    retrieval_context=["New Delhi is the capital of India"],
)

score = answer_relevancy_metric.measure(test_case)
print(score)
print(answer_relevancy_metric)


c:\Users\Girish Kulkarni\OneDrive\Documents\LLM_Testing\.venv\Lib\site-packages\rich\live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

1.0


In [41]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric

context_precision_metric = ContextualPrecisionMetric(model=judge)

test_case = LLMTestCase(
    input="What is the capital of India?",
    actual_output="New Delhi",
    expected_output="New Delhi is the capital of India.",
    retrieval_context=["New Delhi is the capital city of India."]
)

score = context_precision_metric.measure(test_case)

print("Score:", context_precision_metric.score)
print("Success:", context_precision_metric.success)
print("Breakdown:", context_precision_metric.score_breakdown)

Score: 1.0
Success: True
Breakdown: None


In [ ]:
# Run the evaluation and publish the result to Confident AI when CONFIDENT_API_KEY is configured
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

answer_relevancy_metric = AnswerRelevancyMetric(model=judge)

test_case = LLMTestCase(
    input="Capital of India",
    actual_output="Delhi",
    retrieval_context=["New Delhi is the capital of India"],
)


evaluation_result = evaluate(
    test_cases=[test_case],
    metrics=[answer_relevancy_metric]
)
print(evaluation_result)
print("Confident AI link:", evaluation_result.confident_link)


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:8b (Ollama), strict=False, 
async_mode=True)...

In [ ]:
# Optional: log in to Confident AI so evaluate() can publish results.
# The judge still comes from the setup cell (OpenAI or local Ollama).
import os
import deepeval

api_key = os.getenv("CONFIDENT_API_KEY")
if not api_key or "paste_your_key" in api_key:
    print("Skipping Confident AI login. Set CONFIDENT_API_KEY in notebooks/.env to publish results.")
else:
    deepeval.login(api_key=api_key)
    print("Logged in to Confident AI.")

🎉🥳 Congratulations! You've successfully logged in! 🙌

Logged in to Confident AI.


In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

answer_relevancy_metric = AnswerRelevancyMetric(model=judge)

test_case1 = LLMTestCase(
    input="Capital of India",
    actual_output="Delhi",
    retrieval_context=["New Delhi is the capital of India"],
)
test_case2 = LLMTestCase(
    input="Who built GPT models",
    actual_output="Open AI",
    retrieval_context=["Open AI built GPT models"],
)

evaluation_result = evaluate(
    test_cases=[test_case1, test_case2],
    metrics=[answer_relevancy_metric]
)
print(evaluation_result)
print("Confident AI link:", evaluation_result.confident_link)


✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=2 | failed=0                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=14065774;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

Test run saved at deepeval-results\test_run_20260920_101800.json

✓ Done 🎉! View results on 
]8;id=14065777;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9c5dc3002loq0t4ka7pk0b\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9c5dc3002loq0t4ka7pk0b]8;;\

test_results=[TestResult(name='test_case_1', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the question about who built GPT models without any irrelevant statements.', strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00019784999999999998, input_tokens=1043, output_tokens=69, verbose_logs='Statements:\n[\n    "Open AI"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, index=1, multimodal=False, input='Who built GPT models', actual_output='Open AI', expected_output=None, context=None, retrieval_context=['Open AI built GPT models'], turns=None, metadata=None), TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly addressed t

In [ ]:
# Evaluation dataset
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.dataset import EvaluationDataset, Golden

answer_relevancy_metric = AnswerRelevancyMetric(model=judge)

golden = Golden(
    input="Capital of India",
    expected_output="Delhi",
    context=["New Delhi is the capital of India"],
)

dataset = EvaluationDataset()
dataset.add_golden(golden)

dataset


EvaluationDataset(test_cases=[], goldens=[Golden(id=None, input='Capital of India', actual_output=None, expected_output='Delhi', context=['New Delhi is the capital of India'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, token_cost=None, input_token_count=None, output_token_count=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None)], _alias=None, _id=None, _version=None, _multi_turn=False)

In [ ]:
# Creating a testcase from golden

for golden in dataset.goldens:
    test_case = LLMTestCase(
        input=golden.input,
        expected_output=golden.expected_output,
        actual_output= "Delhi",
        retrieval_context=golden.context,
    )
    
    dataset.add_test_case(test_case)
    
    evaluation_result = evaluate(
    test_cases=dataset.test_cases,
    metrics=[answer_relevancy_metric]
)
print(evaluation_result)
print("Confident AI link:", evaluation_result.confident_link)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=14065779;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

Test run saved at deepeval-results\test_run_20260920_101805.json

✓ Done 🎉! View results on 
]8;id=14065782;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9c5gxa004amy0twqezvhlj\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9c5gxa004amy0twqezvhlj]8;;\

test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly addressed the input question about the capital of India without any irrelevant statements.', strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00019725, input_tokens=1039, output_tokens=69, verbose_logs='Statements:\n[\n    "Delhi"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]')], conversational=False, index=0, multimodal=False, input='Capital of India', actual_output='Delhi', expected_output='Delhi', context=None, retrieval_context=['New Delhi is the capital of India'], turns=None, metadata=None)] confident_link='https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9c5gxa004amy0twqezvhlj' test_run_id='cmu9c5gxa004amy0twqezvhlj'
Confident AI link: https://app.confident-ai.com/

In [ ]:
# Create multiple set of datasets
# Create goldens from code and push to confidentAi 


test_data = [
    {
        "input": "What is the capital of India?",
        "expected_output": "New Delhi",
        "context": ["New Delhi is the capital of India."]
    },
    {
        "input": "Who developed the Python programming language?",
        "expected_output": "Guido van Rossum",
        "context": ["Python was created by Guido van Rossum."]
    },
    {
        "input": "What is the largest planet in our solar system?",
        "expected_output": "Jupiter",
        "context": ["Jupiter is the largest planet in the solar system."]
    },
    {
        "input": "What is the boiling point of water?",
        "expected_output": "100°C at standard atmospheric pressure.",
        "context": ["Water boils at 100°C at standard atmospheric pressure."]
    },
    {
        "input": "Who built the GPT models?",
        "expected_output": "OpenAI",
        "context": ["OpenAI developed the GPT family of models."]
    }
]



In [ ]:
# Create the goldens
from deepeval.dataset import EvaluationDataset, Golden
goldens = []

for data in test_data:
    golden = Golden(
        input=data["input"],
        expected_output=data["expected_output"],
        context=data["context"],
    )
    goldens.append(golden)

new_dataset = EvaluationDataset(goldens=goldens)
new_dataset

EvaluationDataset(test_cases=[], goldens=[Golden(id=None, input='What is the capital of India?', actual_output=None, expected_output='New Delhi', context=['New Delhi is the capital of India.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, token_cost=None, input_token_count=None, output_token_count=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(id=None, input='Who developed the Python programming language?', actual_output=None, expected_output='Guido van Rossum', context=['Python was created by Guido van Rossum.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, token_cost=None, input_token_count=None, output_token_count=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(id=None, input='What is the largest planet in our solar system?', actual_out

In [ ]:
new_dataset.push(alias="GoldenDataSet")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=14065784;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/datasets/cmu6azg62000lmh0tfjsbyk9k\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/datasets/cmu6azg62000lmh0tfjsbyk9k]8;;\

In [ ]:
# pull the data from confidnetAI
# confidnetDataSet = EvaluationDataset()
# confidnetDataSet.pull(alias="GoldenDataSet")



In [ ]:
# Optional standalone mock. The next cell defines mock_llm again so you can run it by itself.
def mock_llm(index):
    answers = {
        1: "New Delhi",
        2: "Bangalore",
        3: "Jupiter",
        4: "0°C",
        5: "Google",
    }
    return answers.get(index, "unknown")

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

# Stand-in for your LLM so this cell runs even if the previous cell was skipped.
def mock_llm(index):
    answers = {
        1: "New Delhi",
        2: "Bangalore",
        3: "Jupiter",
        4: "0°C",
        5: "Google",
    }
    return answers.get(index, "unknown")

# Must pass a metric *instance*, not the class AnswerRelevancyMetric.
answer_relevancy_metric = AnswerRelevancyMetric(model=judge)

new_dataset.test_cases.clear()
counter = 1
for golden in new_dataset.goldens:
    test_case = LLMTestCase(
        input=golden.input,
        expected_output=golden.expected_output,
        actual_output=mock_llm(counter),
        retrieval_context=golden.context,
    )
    counter += 1
    new_dataset.add_test_case(test_case)

evaluation_result = evaluate(
    test_cases=new_dataset.test_cases,
    metrics=[answer_relevancy_metric],
)
evaluation_result

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              Who developed the Python programming language?                                       │
│  │     Actual Output:      Bangalore                                                                            │
│  │     Expected Output:    Guido van Rossum                                                                     │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because the output included an          │
│              │                  │       │           │ irrelevant statement about 'Bangalore' that does not      │
│              │                  │       │           │ address the question of who developed the Python          │
│              │                  │       │           │ programming language.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_3                                                                                                 │
│  ├──   Input:              What is the boiling point of water?                                                  │
│  │     Actual Output:      0°C                                                                                  │
│  │     Expected Output:    100°C at standard atmospheric pressure.                                              │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because the response incorrectly        │
│              │                  │       │           │ state

⚠ WARNING: No hyperparameters logged.
» ]8;id=14065786;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

Test run saved at deepeval-results\test_run_20260920_101812.json

✓ Done 🎉! View results on 
]8;id=14065789;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9c5m2h002moq0tb9ylti70\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmu9c5m2h002moq0tb9ylti70]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_1', success=False, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=False, score=0.0, reason="The score is 0.00 because the output included an irrelevant statement about 'Bangalore' that does not address the question of who developed the Python programming language.", strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.000216, input_tokens=1068, output_tokens=93, verbose_logs='Statements:\n[\n    "Bangalore"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "no",\n        "reason": "The statement \'Bangalore\' does not provide any information about the developers of the Python programming language."\n    }\n]')], conversational=False, index=1, multimodal=False, input='Who developed the Python programming language?', actual_output='Bangalore', expected_output='Guido van Rossum', context=None, retrieval_context=['Python was created by Guido van Rossum.'], turns=None, 